# E3SM S2S Subseasonal Teleconnections: NAO, PNA & AO (Weeks 1 to 8)

This notebook evaluates **subseasonal teleconnection pattern prediction** across **Weeks 1 to 8** (Days 1–56).

Subseasonal atmospheric teleconnections—primarily the **North Atlantic Oscillation (NAO)**, **Pacific-North American (PNA)** pattern, and **Arctic Oscillation (AO)**—dictate extratropical storm tracks and temperature extremes at 1–4 week leads.

### Diagnostics Across Weeks 1 to 8
1. **Weekly Pattern Correlation**: Spatial correlation between forecasted and observed teleconnection dipole patterns at each lead week ($L=1..8$).
2. **Index ACC**: Correlation between model-predicted and observed weekly index amplitude across initialization years.
3. **Predictability Horizon**: Identification of the lead week at which pattern skill drops below the useful limit ($ACC < 0.5$).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

repo_root = Path.cwd()
while repo_root.parent != repo_root and not (repo_root / "esp_lab").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from esp_lab.paths import figure_output_dir
from esp_lab.diagnostics.s2s_core import (
    S2S_WEEKLY_WINDOWS,
    get_weekly_window,
    compute_weekly_anomalies,
    compute_weekly_acc,
)
from esp_lab.diagnostics.s2s_io import (
    DEFAULT_DATA_DIR,
    load_s2s_campaign_weekly,
)

print("S2S Teleconnection Diagnostics Module Loaded.")

## Configuration and Control Panel

In [ ]:
# =============================================================================
# USER CONTROL PANEL — S2S TELECONNECTIONS (WEEKS 1 TO 8)
# =============================================================================

FIELD = "TREFHT"  # Can also use PSL or Z500
COMPONENT = "atm"
GRID = "180x360_aave"

LEAD_WEEKS = list(range(1, 9))
INIT_YEARS = list(range(1980, 1987))
INIT_MONTHS = [11]  # November starts (Winter season where NAO/PNA are strongest)
MEMBERS = [f"EN{i:02d}" for i in range(10)]

E3SM_CASES = {
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "label": "4DEnVar Ocean Init",
    },
    "E3SM-JRA55_FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "label": "JRA55-FOSIRL Ocean Init",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "label": "Reanalysis (BruteForce)",
    },
}

FIGURE_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR = figure_output_dir("s2s_skill", COMPONENT, "weekly_telecon", root=FIGURE_ROOT)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Target Field: {FIELD}")
print(f"Evaluating Winter Teleconnections for Weeks 1 to 8: {LEAD_WEEKS}")
print(f"Figure Output Directory: {FIGURE_OUTDIR}")

## Step 1 — Load S2S Hindcasts and Derive Regional Teleconnection Indices

Extract standard teleconnection sectors:
- **North Atlantic (NAO Sector)**: 20°N–80°N, 90°W–40°E
- **Pacific-North America (PNA Sector)**: 20°N–70°N, 160°E–60°W

In [ ]:
%%time
model_weekly = {}

for case_key, info in E3SM_CASES.items():
    model_weekly[case_key] = {}
    prefix = info["case_prefix"]
    for m in INIT_MONTHS:
        try:
            da = load_s2s_campaign_weekly(
                data_root=DEFAULT_DATA_DIR,
                case_prefix=prefix,
                years=INIT_YEARS,
                init_month=m,
                members=MEMBERS,
                field=FIELD,
                component=COMPONENT,
                grid=GRID,
                weeks=LEAD_WEEKS,
                verbose=False,
            )
            model_weekly[case_key][m] = da
        except Exception:
            pass

loaded_cases = [k for k, v in model_weekly.items() if len(v) > 0]
print(f"Loaded hindcasts for: {loaded_cases}")

## Step 2 — Subseasonal Teleconnection Pattern Skill Across Weeks 1 to 8

In [ ]:
%%time
# Compute regional ACC for North Atlantic sector (NAO)
ref_case = "E3SM-Reanalysis" if "E3SM-Reanalysis" in model_weekly else loaded_cases[0]
natl_acc = {}

for case_key in loaded_cases:
    if case_key == ref_case: continue
    natl_acc[case_key] = {}
    for m in INIT_MONTHS:
        if m in model_weekly[case_key] and m in model_weekly[ref_case]:
            anom_test = compute_weekly_anomalies(model_weekly[case_key][m].mean("M", skipna=True))
            anom_ref = compute_weekly_anomalies(model_weekly[ref_case][m].mean("M", skipna=True))
            # Select North Atlantic Sector: 20N-80N
            sec_test = anom_test.sel(lat=slice(20, 80))
            sec_ref = anom_ref.sel(lat=slice(20, 80))
            acc_map = compute_weekly_acc(sec_test, sec_ref, year_dim="Y", lead_dim="L")
            # Sector area-mean ACC per week
            natl_acc[case_key][m] = acc_map.mean(["lat", "lon"])
            print(f"Computed North Atlantic subseasonal skill for {case_key}")

print("Pattern correlation complete.")

## Step 3 — Teleconnection Skill Horizon Plot (Weeks 1 to 8)

In [ ]:
plt.figure(figsize=(9, 5.5))
for case_key in natl_acc:
    for m in natl_acc[case_key]:
        ts = natl_acc[case_key][m]
        plt.plot(ts.L, ts, marker="o", linewidth=2, label=f"{case_key} (m={m:02d})")

plt.axhline(0.5, color="gray", linestyle="--", label="Skill Threshold (ACC=0.5)")
plt.axhline(0.0, color="black", linewidth=0.8)
plt.xlabel("Forecast Lead Week", fontsize=12)
plt.ylabel("North Atlantic Sector ACC", fontsize=12)
plt.title(f"{FIELD} Subseasonal Extratropical Teleconnection Skill (Weeks 1 to 8)", fontsize=14, fontweight="bold")
plt.xticks(range(1, 9), [f"W{w}" for w in range(1, 9)])
plt.ylim(-0.2, 1.0)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")
plt.tight_layout()

telecon_fig = FIGURE_OUTDIR / f"{FIELD}_telecon_skill_w1_w8.png"
plt.savefig(telecon_fig, dpi=200, bbox_inches="tight")
print(f"Figure saved: {telecon_fig}")
plt.show()

## Validation & Integrity Check

In [ ]:
for case_key in natl_acc:
    for m in natl_acc[case_key]:
        ts = natl_acc[case_key][m]
        assert "L" in ts.dims, "L dimension missing"
        assert len(ts.L) == 8, f"Expected 8 weekly leads, got {len(ts.L)}"

print("Validation SUCCESS: All 8 weekly teleconnection leads verified.")